In [1]:
#!/usr/bin/env python3
"""
Comprehensive Statistical Analysis of Post-Hoc Calibration Results
Aggregates results across seeds and computes 95% confidence intervals
"""

import json
import os
from typing import Dict, List, Optional, Tuple, Union
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.stats import ttest_rel, ttest_ind
import warnings
warnings.filterwarnings('ignore')

class CalibrationResultsAnalyzer:
    """Comprehensive analysis of post-hoc calibration results"""
    
    def __init__(self, base_results_dir: str):
        """
        Initialize analyzer
        
        Args:
            base_results_dir: Base directory containing phase2 calibration results
        """
        self.base_dir = Path(base_results_dir)
        self.raw_results: List[Dict] = []
        self.aggregated_results: Optional[pd.DataFrame] = None
        self.results_df: Optional[pd.DataFrame] = None
        
    def load_all_results(self):
        """Load all calibration result files and create master DataFrame"""
        print("📊 Loading calibration results...")
        
        # Find all JSON result files
        json_files = list(self.base_dir.rglob("calibration_*.json"))
        print(f"📁 Found {len(json_files)} result files")
        
        for json_file in json_files:
            try:
                with open(json_file, 'r') as f:
                    data = json.load(f)
                
                # Extract experiment info
                config = data['experiment_config']
                method = config['method']
                dataset = config['dataset']
                model = config['model']
                seed = config['seed']
                
                # Extract calibration results
                results = data['results']
                
                for calibration_method, metrics in results.items():
                    # Skip if error occurred
                    if 'error' in metrics:
                        print(f"⚠️  Skipping {method}|{dataset}|{model}|seed{seed}|{calibration_method}: {metrics['error']}")
                        continue
                    
                    # Extract metrics
                    self.raw_results.append({
                        'training_method': method,
                        'dataset': dataset,
                        'model': model,
                        'seed': seed,
                        'calibration_method': calibration_method,
                        'accuracy': metrics.get('accuracy', 0),
                        'ece': metrics.get('ece', 1.0),
                        'num_samples': metrics.get('num_samples', 0),
                        'calibration_time': metrics.get('calibration_time', 0),
                        'file_path': str(json_file)
                    })
                    
            except Exception as e:
                print(f"❌ Error loading {json_file}: {e}")
                continue
        
        # Convert to DataFrame
        self.results_df = pd.DataFrame(self.raw_results)
        print(f"✅ Loaded {len(self.raw_results)} calibration results")
        
        if len(self.results_df) > 0:
            print("\n📋 Data Summary:")
            print(f"   Training methods: {sorted(self.results_df['training_method'].unique())}")
            print(f"   Datasets: {sorted(self.results_df['dataset'].unique())}")
            print(f"   Models: {sorted(self.results_df['model'].unique())}")
            print(f"   Seeds: {sorted(self.results_df['seed'].unique())}")
            print(f"   Calibration methods: {sorted(self.results_df['calibration_method'].unique())}")
        
        return self.results_df
    
    def aggregate_results(self):
        """Aggregate results across seeds and compute statistics"""
        if self.results_df is None or len(self.results_df) == 0:
            print("❌ No data loaded. Run load_all_results() first.")
            return None
        
        print("\n🔢 Computing aggregated statistics...")
        
        # Group by all factors except seed
        grouped = self.results_df.groupby([
            'training_method', 'dataset', 'model', 'calibration_method'
        ])
        
        stats_list = []
        
        for (training_method, dataset, model, cal_method), group in grouped:
            if len(group) < 1:
                continue
            
            # Calculate statistics for ECE and calibration time
            for metric in ['ece', 'calibration_time', 'accuracy']:
                values = group[metric].values
                n = len(values)
                
                if n == 0:
                    continue
                
                mean_val = np.mean(values)
                std_val = np.std(values, ddof=1) if n > 1 else 0
                se_val = std_val / np.sqrt(n) if n > 1 else 0
                
                # 95% confidence interval using t-distribution
                if n > 1:
                    t_val = stats.t.ppf(0.975, df=n-1)
                    ci_lower = mean_val - t_val * se_val
                    ci_upper = mean_val + t_val * se_val
                else:
                    ci_lower = mean_val
                    ci_upper = mean_val
                
                stats_list.append({
                    'training_method': training_method,
                    'dataset': dataset,
                    'model': model,
                    'calibration_method': cal_method,
                    'metric': metric,
                    'n_seeds': n,
                    'mean': mean_val,
                    'std': std_val,
                    'se': se_val,
                    'ci_lower': ci_lower,
                    'ci_upper': ci_upper,
                    'seeds': sorted(group['seed'].tolist())
                })
        
        self.aggregated_results = pd.DataFrame(stats_list)
        print(f"✅ Computed statistics for {len(stats_list)} combinations")
        
        return self.aggregated_results
    
    def create_structured_analysis_tables(self) -> Dict[str, pd.DataFrame]:
        """Create structured analysis tables to answer key research questions"""
        
        if self.aggregated_results is None:
            self.aggregate_results()
        
        # 1. BEST CALIBRATION METHOD PER MODEL/DATASET/TRAINING_METHOD
        ece_data = self.aggregated_results[self.aggregated_results['metric'] == 'ece'].copy()
        
        # Find best calibration method for each combination
        best_calibration = ece_data.loc[ece_data.groupby([
            'training_method', 'dataset', 'model'
        ])['mean'].idxmin()].copy()
        
        best_calibration_table = best_calibration[[
            'training_method', 'dataset', 'model', 'calibration_method',
            'mean', 'std', 'se', 'ci_lower', 'ci_upper', 'n_seeds'
        ]].rename(columns={
            'mean': 'best_ece_mean',
            'std': 'best_ece_std', 
            'se': 'best_ece_se',
            'ci_lower': 'best_ece_ci_lower',
            'ci_upper': 'best_ece_ci_upper',
            'calibration_method': 'best_calibration_method'
        })
        
        # 2. UNCALIBRATED PERFORMANCE COMPARISON
        uncalibrated_data = ece_data[ece_data['calibration_method'] == 'Uncalibrated'].copy()
        uncalibrated_table = uncalibrated_data[[
            'training_method', 'dataset', 'model', 'mean', 'std', 'se', 
            'ci_lower', 'ci_upper', 'n_seeds'
        ]].rename(columns={
            'mean': 'uncalibrated_ece_mean',
            'std': 'uncalibrated_ece_std',
            'se': 'uncalibrated_ece_se', 
            'ci_lower': 'uncalibrated_ece_ci_lower',
            'ci_upper': 'uncalibrated_ece_ci_upper'
        }).sort_values('uncalibrated_ece_mean')
        
        # 3. SEED COUNT SUMMARY
        seed_summary = self.results_df.groupby([
            'training_method', 'dataset', 'model', 'calibration_method'
        ]).agg({
            'seed': ['count', 'nunique', lambda x: sorted(list(x))],
            'ece': ['mean', 'std', 'min', 'max'],
            'accuracy': ['mean', 'std'],
            'calibration_time': ['mean', 'std']
        }).round(4)
        
        # Flatten column names
        seed_summary.columns = [f"{col[0]}_{col[1]}" for col in seed_summary.columns]
        seed_summary = seed_summary.reset_index()
        seed_summary.rename(columns={
            'seed_count': 'total_experiments',
            'seed_nunique': 'unique_seeds',
            'seed_<lambda>': 'seed_list'
        }, inplace=True)
        
        # 4. COMPREHENSIVE COMPARISON TABLE
        # Merge best calibration with uncalibrated performance
        comparison_table = best_calibration_table.merge(
            uncalibrated_table[['training_method', 'dataset', 'model', 'uncalibrated_ece_mean', 'uncalibrated_ece_std']],
            on=['training_method', 'dataset', 'model'],
            how='left'
        )
        
        # Calculate improvement
        comparison_table['ece_improvement'] = (
            comparison_table['uncalibrated_ece_mean'] - comparison_table['best_ece_mean']
        )
        comparison_table['ece_improvement_percent'] = (
            comparison_table['ece_improvement'] / comparison_table['uncalibrated_ece_mean'] * 100
        )
        
        # Sort by improvement
        comparison_table = comparison_table.sort_values('ece_improvement', ascending=False)
        
        # 5. DETAILED PERFORMANCE MATRIX
        detailed_matrix = ece_data.pivot_table(
            index=['training_method', 'dataset', 'model'],
            columns='calibration_method',
            values='mean',
            aggfunc='first'
        ).round(4)
        
        # Add ranking for each row (1 = best)
        detailed_rankings = detailed_matrix.rank(axis=1, method='min')
        
        return {
            'best_calibration_summary': best_calibration_table,
            'uncalibrated_performance': uncalibrated_table, 
            'seed_count_summary': seed_summary,
            'comprehensive_comparison': comparison_table,
            'detailed_ece_matrix': detailed_matrix,
            'detailed_rankings': detailed_rankings
        }

    def create_significance_summary_table(self):
        """Create structured statistical significance summary"""
        
        significance_df = self.perform_statistical_tests()
        
        if significance_df is None or len(significance_df) == 0:
            return None
        
        # Filter for significant results and add interpretations
        significant_results = significance_df[
            (significance_df['significant'] == True) & 
            (~significance_df['p_value'].isna())
        ].copy()
        
        # Add effect size interpretation
        def interpret_effect_size(d):
            if pd.isna(d):
                return 'Unknown'
            elif abs(d) < 0.2:
                return 'Negligible'
            elif abs(d) < 0.5:
                return 'Small'
            elif abs(d) < 0.8:
                return 'Medium'
            else:
                return 'Large'
        
        significant_results['effect_size_interpretation'] = significant_results['cohens_d'].apply(interpret_effect_size)
        
        # Calculate improvement percentage
        significant_results['improvement_percent'] = (
            (significant_results['method1_mean'] - significant_results['method2_mean']) / 
            significant_results['method1_mean'] * 100
        )
        
        # Sort by effect size
        significant_results = significant_results.sort_values('cohens_d', ascending=False, key=abs)
        
        return significant_results[[
            'training_method', 'dataset', 'model', 'method1', 'method2',
            'method1_mean', 'method2_mean', 'improvement_percent',
            'p_value', 'cohens_d', 'effect_size_interpretation',
            'method1_n', 'method2_n', 'test_type'
        ]]

    def export_all_tables(self, output_dir: str = "./analysis_tables"):
        """Export all analysis tables to CSV files"""
        
        os.makedirs(output_dir, exist_ok=True)
        
        print(f"\n📊 Creating structured analysis tables in {output_dir}/...")
        
        # Create structured tables
        tables = self.create_structured_analysis_tables()
        significance_table = self.create_significance_summary_table()
        
        # Export each table
        exports = {}
        
        for table_name, table_df in tables.items():
            if table_df is not None and len(table_df) > 0:
                output_path = f"{output_dir}/{table_name}.csv"
                table_df.to_csv(output_path, index=False)
                exports[table_name] = output_path
                print(f"   ✅ {table_name}: {len(table_df)} rows → {output_path}")
        
        # Export significance table
        if significance_table is not None and len(significance_table) > 0:
            sig_path = f"{output_dir}/statistical_significance_summary.csv"
            significance_table.to_csv(sig_path, index=False)
            exports['significance_summary'] = sig_path
            print(f"   ✅ significance_summary: {len(significance_table)} rows → {sig_path}")
        
        # Export raw aggregated results for further analysis
        if self.aggregated_results is not None:
            agg_path = f"{output_dir}/aggregated_results_raw.csv"
            self.aggregated_results.to_csv(agg_path, index=False)
            exports['aggregated_raw'] = agg_path
            print(f"   ✅ aggregated_raw: {len(self.aggregated_results)} rows → {agg_path}")
        
        # Export raw results
        if self.results_df is not None:
            raw_path = f"{output_dir}/raw_results.csv"
            self.results_df.to_csv(raw_path, index=False)
            exports['raw_results'] = raw_path
            print(f"   ✅ raw_results: {len(self.results_df)} rows → {raw_path}")
        
        return exports

    def print_key_insights(self):
        """Print key insights in a structured format"""
        
        tables = self.create_structured_analysis_tables()
        
        print("\n" + "="*80)
        print("🔍 KEY RESEARCH INSIGHTS")
        print("="*80)
        
        # 1. Best uncalibrated models
        print("\n📊 BEST UNCALIBRATED MODELS (by ECE):")
        print("-" * 50)
        uncalibrated = tables['uncalibrated_performance'].head(5)
        for _, row in uncalibrated.iterrows():
            print(f"   {row['training_method']:20s} | {row['dataset']:8s} | {row['model']:12s} | "
                f"ECE: {row['uncalibrated_ece_mean']:.4f} ± {row['uncalibrated_ece_se']*1.96:.4f}")
        
        # 2. Biggest improvements from calibration
        print("\n🏆 BIGGEST IMPROVEMENTS FROM CALIBRATION:")
        print("-" * 50)
        improvements = tables['comprehensive_comparison'].head(5)
        for _, row in improvements.iterrows():
            print(f"   {row['training_method']:20s} | {row['dataset']:8s} | {row['model']:12s}")
            print(f"      Uncalibrated: {row['uncalibrated_ece_mean']:.4f}")
            print(f"      Best ({row['best_calibration_method']}): {row['best_ece_mean']:.4f}")
            print(f"      Improvement: {row['ece_improvement_percent']:.1f}%")
            print()
        
        # 3. Most common best calibration methods
        print("🎯 MOST EFFECTIVE CALIBRATION METHODS:")
        print("-" * 50)
        method_counts = tables['best_calibration_summary']['best_calibration_method'].value_counts()
        for method, count in method_counts.head(5).items():
            percentage = count / len(tables['best_calibration_summary']) * 100
            print(f"   {method:30s}: {count:2d} times ({percentage:4.1f}%)")
        
        # 4. Seed coverage summary
        print("\n📈 EXPERIMENT COVERAGE:")
        print("-" * 50)
        seed_summary = tables['seed_count_summary']
        total_configs = len(seed_summary)
        complete_configs = len(seed_summary[seed_summary['unique_seeds'] >= 3])
        print(f"   Total configurations: {total_configs}")
        print(f"   With ≥3 seeds: {complete_configs} ({complete_configs/total_configs*100:.1f}%)")
        
        avg_seeds = seed_summary['unique_seeds'].mean()
        print(f"   Average seeds per config: {avg_seeds:.1f}")

    def create_summary_tables(self):
        """Create publication-ready summary tables"""
        if self.aggregated_results is None:
            print("❌ No aggregated results. Run aggregate_results() first.")
            return None
        
        print("\n📊 Creating summary tables...")
        
        # ECE Summary Table
        ece_data = self.aggregated_results[
            self.aggregated_results['metric'] == 'ece'
        ].copy()
        
        # Create formatted results string with CI
        ece_data['result_str'] = ece_data.apply(
            lambda row: f"{row['mean']:.4f} ± {1.96*row['se']:.4f}" if row['se'] > 0 
            else f"{row['mean']:.4f}", axis=1
        )
        
        # Pivot table for ECE
        ece_pivot = ece_data.pivot_table(
            index=['training_method', 'dataset', 'model'],
            columns='calibration_method',
            values='result_str',
            aggfunc='first'
        )
        
        # Time Summary Table
        time_data = self.aggregated_results[
            self.aggregated_results['metric'] == 'calibration_time'
        ].copy()
        
        time_data['result_str'] = time_data.apply(
            lambda row: f"{row['mean']:.2f}s ± {1.96*row['se']:.2f}s" if row['se'] > 0 
            else f"{row['mean']:.2f}s", axis=1
        )
        
        time_pivot = time_data.pivot_table(
            index=['training_method', 'dataset', 'model'],
            columns='calibration_method',
            values='result_str',
            aggfunc='first'
        )
        
        return {
            'ece_summary': ece_pivot,
            'time_summary': time_pivot,
            'ece_data': ece_data,
            'time_data': time_data
        }
    
    def perform_statistical_tests(self):
        """Perform statistical significance tests"""
        if self.results_df is None:
            print("❌ No data loaded.")
            return None
        
        print("\n🔬 Performing statistical significance tests...")
        
        significance_results = []
        
        # Group by training method, dataset, model
        for (training_method, dataset, model), group in self.results_df.groupby([
            'training_method', 'dataset', 'model'
        ]):
            
            # Get data for each calibration method
            cal_methods = group['calibration_method'].unique()
            
            if len(cal_methods) < 2:
                continue
            
            # Prepare data for pairwise comparisons
            method_data = {}
            for cal_method in cal_methods:
                method_group = group[group['calibration_method'] == cal_method]
                if len(method_group) >= 1:  # At least 1 measurement
                    method_data[cal_method] = method_group['ece'].values
            
            # Pairwise t-tests for ECE (lower is better)
            methods = list(method_data.keys())
            for i, method1 in enumerate(methods):
                for method2 in methods[i+1:]:
                    if method1 in method_data and method2 in method_data:
                        ece1 = method_data[method1]
                        ece2 = method_data[method2]
                        
                        if len(ece1) > 1 and len(ece2) > 1:
                            # Independent t-test (different seeds)
                            t_stat, p_val = stats.ttest_ind(ece1, ece2)
                            test_type = 'independent'
                        elif len(ece1) == len(ece2) and len(ece1) > 1:
                            # Paired t-test (same seeds)
                            t_stat, p_val = stats.ttest_rel(ece1, ece2)
                            test_type = 'paired'
                        else:
                            # Single measurement comparison
                            t_stat, p_val = np.nan, np.nan
                            test_type = 'single_measurement'
                        
                        # Effect size (Cohen's d)
                        if len(ece1) > 1 or len(ece2) > 1:
                            pooled_std = np.sqrt(((len(ece1)-1)*np.var(ece1, ddof=1) + 
                                                (len(ece2)-1)*np.var(ece2, ddof=1)) / 
                                               (len(ece1) + len(ece2) - 2))
                            cohens_d = (np.mean(ece1) - np.mean(ece2)) / pooled_std if pooled_std > 0 else 0
                        else:
                            cohens_d = np.nan
                        
                        significance_results.append({
                            'training_method': training_method,
                            'dataset': dataset,
                            'model': model,
                            'method1': method1,
                            'method2': method2,
                            'test_type': test_type,
                            't_statistic': t_stat,
                            'p_value': p_val,
                            'cohens_d': cohens_d,
                            'method1_mean': np.mean(ece1),
                            'method2_mean': np.mean(ece2),
                            'method1_n': len(ece1),
                            'method2_n': len(ece2),
                            'significant': p_val < 0.05 if not np.isnan(p_val) else False
                        })
        
        return pd.DataFrame(significance_results)
    
    def create_visualizations(self, save_dir: str = "./analysis_results"):
        """Create comprehensive visualizations"""
        if self.results_df is None:
            print("❌ No data loaded.")
            return
        
        os.makedirs(save_dir, exist_ok=True)
        
        # Set style
        plt.style.use('default')
        sns.set_palette("husl")
        
        print(f"\n📈 Creating visualizations in {save_dir}/...")
        
        # 1. ECE Comparison Box Plot
        plt.figure(figsize=(16, 10))
        
        # Filter out methods with very few results for cleaner plots
        method_counts = self.results_df['calibration_method'].value_counts()
        common_methods = method_counts[method_counts >= 3].index
        
        plot_data = self.results_df[
            self.results_df['calibration_method'].isin(common_methods)
        ]
        
        sns.boxplot(
            data=plot_data,
            x='calibration_method',
            y='ece',
            hue='dataset'
        )
        
        plt.title('Expected Calibration Error (ECE) Comparison', fontsize=16, fontweight='bold')
        plt.xlabel('Calibration Method', fontsize=12)
        plt.ylabel('Expected Calibration Error (ECE)', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.legend(title='Dataset', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.savefig(f"{save_dir}/ece_comparison_boxplot.png", dpi=300, bbox_inches='tight')
        plt.close()
        
        # 2. Mean ECE Bar Plot with Error Bars
        if self.aggregated_results is not None:
            ece_stats = self.aggregated_results[self.aggregated_results['metric'] == 'ece']
            
            plt.figure(figsize=(16, 8))
            
            # Create subplot for each dataset
            datasets = ece_stats['dataset'].unique()
            n_datasets = len(datasets)
            
            fig, axes = plt.subplots(1, n_datasets, figsize=(6*n_datasets, 8), sharey=True)
            if n_datasets == 1:
                axes = [axes]
            
            for i, dataset in enumerate(datasets):
                dataset_data = ece_stats[ece_stats['dataset'] == dataset]
                
                # Get unique calibration methods for this dataset
                cal_methods = dataset_data['calibration_method'].unique()
                x_pos = range(len(cal_methods))
                
                means = []
                errors = []
                for method in cal_methods:
                    method_data = dataset_data[dataset_data['calibration_method'] == method]
                    if len(method_data) > 0:
                        means.append(method_data['mean'].iloc[0])
                        errors.append(1.96 * method_data['se'].iloc[0])  # 95% CI
                    else:
                        means.append(0)
                        errors.append(0)
                
                axes[i].bar(x_pos, means, yerr=errors, capsize=5, alpha=0.7)
                axes[i].set_title(f'{dataset.upper()}', fontsize=14, fontweight='bold')
                axes[i].set_xlabel('Calibration Method')
                if i == 0:
                    axes[i].set_ylabel('Expected Calibration Error (ECE)')
                axes[i].set_xticks(x_pos)
                axes[i].set_xticklabels(cal_methods, rotation=45, ha='right')
                axes[i].grid(True, alpha=0.3)
            
            plt.suptitle('Mean ECE by Dataset and Calibration Method (95% CI)', fontsize=16)
            plt.tight_layout()
            plt.savefig(f"{save_dir}/mean_ece_barplot.png", dpi=300, bbox_inches='tight')
            plt.close()
        
        # 3. Calibration Time vs ECE Scatter Plot
        plt.figure(figsize=(12, 8))
        
        for dataset in self.results_df['dataset'].unique():
            dataset_data = self.results_df[self.results_df['dataset'] == dataset]
            plt.scatter(
                dataset_data['calibration_time'],
                dataset_data['ece'],
                label=dataset,
                alpha=0.7,
                s=60
            )
        
        plt.xlabel('Calibration Time (seconds)', fontsize=12)
        plt.ylabel('Expected Calibration Error (ECE)', fontsize=12)
        plt.title('ECE vs Calibration Time Trade-off', fontsize=16, fontweight='bold')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.xscale('log')  # Log scale for time
        plt.savefig(f"{save_dir}/ece_vs_time_scatter.png", dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"✅ Visualizations saved to {save_dir}/")
    
    def generate_report(self, output_file: str = "calibration_analysis_report.md"):
        """Generate comprehensive analysis report"""
        print(f"\n📄 Generating comprehensive report: {output_file}")
        
        # Load data if not already loaded
        if self.results_df is None:
            self.load_all_results()
        
        if self.aggregated_results is None:
            self.aggregate_results()
        
        # Get summary tables and statistical tests
        summary_tables = self.create_summary_tables()
        significance_df = self.perform_statistical_tests()
        
        # Generate report
        report_lines = [
            "# Post-Hoc Calibration Analysis Report",
            f"Generated on: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}",
            "",
            "## Executive Summary",
            f"- **Total experiments analyzed:** {len(self.results_df)}",
            f"- **Training methods:** {', '.join(sorted(self.results_df['training_method'].unique()))}",
            f"- **Datasets:** {', '.join(sorted(self.results_df['dataset'].unique()))}",
            f"- **Models:** {', '.join(sorted(self.results_df['model'].unique()))}",
            f"- **Seeds analyzed:** {', '.join(map(str, sorted(self.results_df['seed'].unique())))}",
            f"- **Calibration methods:** {len(self.results_df['calibration_method'].unique())}",
            "",
        ]
        
        # Add ECE summary table
        if summary_tables and 'ece_summary' in summary_tables:
            report_lines.extend([
                "## Expected Calibration Error (ECE) Summary",
                "Mean ± 95% Confidence Interval",
                "",
                "```",
                str(summary_tables['ece_summary']),
                "```",
                "",
            ])
        
        # Add calibration time summary
        if summary_tables and 'time_summary' in summary_tables:
            report_lines.extend([
                "## Calibration Time Summary",
                "Mean ± 95% Confidence Interval",
                "",
                "```",
                str(summary_tables['time_summary']),
                "```",
                "",
            ])
        
        # Add best performing methods
        if self.aggregated_results is not None:
            ece_stats = self.aggregated_results[self.aggregated_results['metric'] == 'ece']
            best_methods = ece_stats.nsmallest(10, 'mean')[
                ['training_method', 'dataset', 'model', 'calibration_method', 'mean', 'ci_lower', 'ci_upper', 'n_seeds']
            ]
            
            report_lines.extend([
                "## Top 10 Best Performing Methods (Lowest ECE)",
                "",
            ])
            
            for i, (_, row) in enumerate(best_methods.iterrows(), 1):
                report_lines.append(
                    f"{i}. **{row['training_method']} + {row['calibration_method']}** "
                    f"({row['dataset']}, {row['model']}): "
                    f"ECE = {row['mean']:.4f} "
                    f"(95% CI: [{row['ci_lower']:.4f}, {row['ci_upper']:.4f}], "
                    f"n={row['n_seeds']} seeds)"
                )
            
            report_lines.append("")
        
        # Add statistical significance results
        if significance_df is not None and len(significance_df) > 0:
            report_lines.extend([
                "## Statistical Significance Tests",
                "",
                "### Significant Differences (p < 0.05)",
                "",
            ])
            
            significant_tests = significance_df[
                (significance_df['significant'] == True) & 
                (~significance_df['p_value'].isna())
            ].sort_values('p_value')
            
            for _, row in significant_tests.head(20).iterrows():
                effect_size = "Large" if abs(row['cohens_d']) > 0.8 else "Medium" if abs(row['cohens_d']) > 0.5 else "Small"
                improvement = ((row['method1_mean'] - row['method2_mean']) / row['method1_mean']) * 100
                
                report_lines.extend([
                    f"**{row['training_method']} | {row['dataset']} | {row['model']}**",
                    f"- {row['method1']} vs {row['method2']}",
                    f"- Mean ECE: {row['method1_mean']:.4f} vs {row['method2_mean']:.4f}",
                    f"- Improvement: {improvement:+.1f}%",
                    f"- p-value: {row['p_value']:.4f}, Effect size: {effect_size} (Cohen's d: {row['cohens_d']:.3f})",
                    ""
                ])
        
        # Write report
        with open(output_file, 'w') as f:
            f.write('\n'.join(report_lines))
        
        print(f"✅ Analysis report saved to {output_file}")
        
        return {
            'summary_tables': summary_tables,
            'significance_tests': significance_df,
            'best_methods': best_methods if 'best_methods' in locals() else None
        }

def main():
    """Enhanced main analysis function with structured outputs"""
    
    # Initialize analyzer
    base_results_dir = "/home/ptamar/geometric-internal-calibration/aaai_full_experiments/phase2_calibration/results"
    
    analyzer = CalibrationResultsAnalyzer(base_results_dir)
    
    # Load and analyze results
    print("🔍 Loading calibration results...")
    df = analyzer.load_all_results()
    
    if len(df) == 0:
        print("❌ No results found. Check the results directory path.")
        return
    
    print("📊 Computing aggregated statistics...")
    stats_df = analyzer.aggregate_results()
    
    # Export structured tables
    print("📁 Exporting structured analysis tables...")
    exported_files = analyzer.export_all_tables("./calibration_analysis_tables")
    
    # Create visualizations
    print("📈 Creating visualizations...")
    analyzer.create_visualizations("./calibration_analysis_plots")
    
    # Print key insights
    analyzer.print_key_insights()
    
    print("\n" + "="*80)
    print("📁 ANALYSIS OUTPUTS")
    print("="*80)
    print("\nStructured Tables (CSV):")
    for table_name, file_path in exported_files.items():
        print(f"   📄 {table_name}: {file_path}")
    
    print("\nVisualization Plots:")
    print("   📊 ./calibration_analysis_plots/")
    
    print(f"\n🔬 Next Steps:")
    print(f"   1. Open 'comprehensive_comparison.csv' to see overall best methods")
    print(f"   2. Check 'uncalibrated_performance.csv' for baseline comparisons") 
    print(f"   3. Review 'statistical_significance_summary.csv' for statistical tests")
    print(f"   4. Use 'detailed_ece_matrix.csv' for method-by-method comparisons")

if __name__ == "__main__":
    main()



🔍 Loading calibration results...
📊 Loading calibration results...
📁 Found 113 result files
⚠️  Skipping baseline_cross_entropy|cifar100|densenet121|seed14|Geometric: CUDA out of memory. Tried to allocate 4.88 GiB. GPU 0 has a total capacity of 23.60 GiB of which 4.47 GiB is free. Including non-PyTorch memory, this process has 19.12 GiB memory in use. Of the allocated memory 18.76 GiB is allocated by PyTorch, and 52.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
⚠️  Skipping baseline_cross_entropy|cifar100|densenet121|seed13|Geometric: CUDA out of memory. Tried to allocate 4.88 GiB. GPU 0 has a total capacity of 23.60 GiB of which 4.47 GiB is free. Including non-PyTorch memory, this process has 19.12 GiB memory in use. Of the allocated memory 18.76

<Figure size 1600x800 with 0 Axes>

In [2]:
import pandas as pd

# Which calibration method was best for each model?
best_cal = pd.read_csv('./calibration_analysis_tables/comprehensive_comparison.csv')
print(best_cal[['training_method', 'dataset', 'model', 'best_calibration_method', 'ece_improvement_percent']])

# Which model was best uncalibrated?
uncal = pd.read_csv('./calibration_analysis_tables/uncalibrated_performance.csv')
print("Best uncalibrated model:")
print(uncal.iloc[0])

# Seed coverage per configuration
seeds = pd.read_csv('./calibration_analysis_tables/seed_count_summary.csv')
print(seeds[['training_method', 'dataset', 'model', 'unique_seeds']].head(10))

                training_method   dataset        model  \
0          augmix_constellation  cifar100     resnet18   
1        baseline_cross_entropy  cifar100  densenet121   
2                        augmix      svhn     resnet50   
3        baseline_cross_entropy  cifar100     resnet50   
4                        augmix   cifar10     resnet50   
5                 baseline_mmce  cifar100     resnet50   
6        baseline_cross_entropy   cifar10     resnet50   
7        baseline_mmce_weighted  cifar100     resnet50   
8        baseline_cross_entropy   cifar10  densenet121   
9                 baseline_mmce  cifar100     resnet18   
10               baseline_brier  cifar100     resnet18   
11      baseline_focal_adaptive  cifar100  densenet121   
12  geometric_focal_calibration  cifar100     resnet18   
13       baseline_mmce_weighted  cifar100     resnet18   
14                constellation   cifar10     resnet18   
15                       augmix      svhn     resnet18   
16            